# Final System — V3-C + Focal Loss + Token-Level Cross-Attention + Contrastive Loss


In [1]:
!pip install transformers scikit-learn pandas torch tqdm --quiet
print("Libraries installed.")

Libraries installed.


In [2]:
import os, re, random, warnings, itertools
import pandas as pd, numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from IPython.display import display, HTML
import time
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

DATA_DIR   = "/kaggle/input/datasets/pranavsai4205/input-changed/dataset"
OUTPUT_DIR = "/kaggle/working"

# ── Hyperparameters (Change 5: epochs↑, LR↓, freeze↑; Change 6: clauses) ──
BERT_MODEL      = "distilbert-base-uncased"
MAX_LEN         = 256
STANCE_MAX_LEN  = 128
Q_MAX_LEN       = 64          # token-level Q encoding window
MAX_CLAUSES     = 4           # Change 6: was 6
CLAUSE_LEN      = 96          # Change 6: was 64
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
EPOCHS          = 7           # Change 5: was 5
PATIENCE        = 3
DROPOUT         = 0.3
WARMUP_RATIO    = 0.1
FREEZE_EPOCHS   = 3           # Change 5: was 2
LR_EVASION      = 5e-6        # Change 5: was 1e-5
LR_STANCE       = 1e-5        # Change 5: was 2e-5
LABEL_SMOOTH    = 0.1
FOCAL_GAMMA     = 3.0         # Change 2: Focal Loss gamma (bumped 2→3 for stronger PE focus)
CONTRASTIVE_TAU = 0.1         # Change 7: temperature for contrastive loss
CONTRASTIVE_W   = 0.10        # Change 7: weight of contrastive term (bumped 0.05→0.10)

EVASION_LABELS   = ["Non-Evasive", "Partially Evasive", "Evasive"]
EVASION_LABEL2ID = {l: i for i, l in enumerate(EVASION_LABELS)}
STANCE_LABELS    = ["FAVOR", "AGAINST"]
STANCE_LABEL2ID  = {l: i for i, l in enumerate(STANCE_LABELS)}

EVASION_CSV       = f"{DATA_DIR}/QEvasion_train.csv"
STANCE_TRAIN_CSVS = [f"{DATA_DIR}/raw_train_biden.csv",f"{DATA_DIR}/raw_train_trump.csv",f"{DATA_DIR}/raw_train_bernie.csv"]
STANCE_VAL_CSVS   = [f"{DATA_DIR}/raw_val_biden.csv",  f"{DATA_DIR}/raw_val_trump.csv",  f"{DATA_DIR}/raw_val_bernie.csv"]
STANCE_TEST_CSVS  = [f"{DATA_DIR}/raw_test_biden.csv", f"{DATA_DIR}/raw_test_trump.csv", f"{DATA_DIR}/raw_test_bernie.csv"]

FULL_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model_v3c_final.pt")

print("Config loaded.")
print(f"  Architecture : V3-C + TokenCrossAttn + FocalLoss(γ=3) + ContrastiveLoss(w=0.10) + LenFeat + NaNFix + 3:1sampling")
print(f"  Batch:{BATCH_SIZE}  GradAccum:{GRAD_ACCUM}  EffBatch:{BATCH_SIZE*GRAD_ACCUM}")
print(f"  Epochs:{EPOCHS}  Patience:{PATIENCE}  Freeze:{FREEZE_EPOCHS}")
print(f"  LR_evasion:{LR_EVASION}  LR_stance:{LR_STANCE}")
print(f"  MAX_CLAUSES:{MAX_CLAUSES}  CLAUSE_LEN:{CLAUSE_LEN}")
print(f"  Focal gamma:{FOCAL_GAMMA}  Contrastive tau:{CONTRASTIVE_TAU} w:{CONTRASTIVE_W}")

Device: cuda
GPU: Tesla T4
Config loaded.
  Architecture : V3-C + TokenCrossAttn + FocalLoss(γ=3) + ContrastiveLoss(w=0.10) + LenFeat + NaNFix + 3:1sampling
  Batch:16  GradAccum:2  EffBatch:32
  Epochs:7  Patience:3  Freeze:3
  LR_evasion:5e-06  LR_stance:1e-05
  MAX_CLAUSES:4  CLAUSE_LEN:96
  Focal gamma:3.0  Contrastive tau:0.1 w:0.1


In [3]:
# ─────────────────────────────────────────────────────────────
# AUGMENTATION — 5 strategies on PE class only (8x)
# ─────────────────────────────────────────────────────────────

SYNONYM_MAP = {
    "question":["query","inquiry"],"answer":["response","reply"],
    "policy":["plan","strategy"],"government":["administration","authority"],
    "issue":["matter","concern","problem"],"important":["crucial","significant","vital"],
    "believe":["think","consider","feel"],"people":["citizens","individuals","public"],
    "country":["nation","state"],"support":["back","endorse","advocate"],
    "change":["shift","reform","alter"],"need":["require","must"],
    "work":["function","operate","effort"],"make":["create","produce","form"],
    "good":["beneficial","positive","effective"],"said":["stated","noted","mentioned"],
    "know":["understand","recognize","realize"],"think":["believe","consider","feel"],
    "want":["desire","seek","aim"],"time":["period","moment","point"],
    "new":["recent","fresh","updated"],"political":["governmental","civic"],
    "major":["significant","key","primary"],"different":["various","distinct","alternative"],
    "president":["commander-in-chief","leader","executive"],
    "congress":["legislature","lawmakers","senate"],
    "economy":["market","financial system","fiscal situation"],
    "certainly":["absolutely","definitely","of course"],
    "perhaps":["possibly","maybe","one could argue"],
    "however":["that said","nevertheless","on the other hand"],
    "basically":["fundamentally","essentially","at its core"],
}

EVASION_PHRASES = {
    "i think we should":"it is my view that we ought to",
    "i believe that":"it is my conviction that",
    "we need to":"it is essential that we",
    "i want to":"my intention is to",
    "we are working on":"efforts are underway to",
    "i don't think":"it is not my view that",
    "the fact is":"what we know is",
    "i would say":"my position would be",
    "let me be clear":"to make this absolutely clear",
    "the truth is":"the reality of the situation is",
    "we have to":"it is necessary that we",
    "i am committed to":"my commitment remains to",
    "going forward":"in the coming period",
    "at the end of the day":"ultimately",
    "we must":"it is imperative that we",
    "i am confident":"i have full confidence",
}

STOPWORDS = {"the","a","an","is","are","was","were","be","been","being",
             "have","has","had","do","does","did","will","would","could",
             "should","may","might","to","of","in","for","on","with",
             "at","by","from","as","into","i","we","it","this","that"}

def aug_synonym_replace(text, n=3, seed=None):
    if seed is not None: random.seed(seed)
    words = text.split()
    rep = [(i,w.lower().strip('.,!?;:\"\'()')) for i,w in enumerate(words)
           if w.lower().strip('.,!?;:\"\'()') in SYNONYM_MAP]
    if not rep: return text
    chosen = random.sample(rep, min(n, len(rep)))
    nw = words[:]
    for i,w in chosen:
        syn = random.choice(SYNONYM_MAP[w])
        if words[i][0].isupper(): syn=syn[0].upper()+syn[1:]
        nw[i] = syn
    return " ".join(nw)

def aug_structure_swap(text, n=2, seed=None):
    if seed is not None: random.seed(seed)
    clauses = re.split(r'(?<=[.!?,;])\s+', text)
    clauses = [c for c in clauses if len(c.split()) >= 4]
    if not clauses:
        words = text.split()
        if len(words) < 4: return text
        for _ in range(n):
            i,j = random.sample(range(len(words)),2)
            words[i],words[j] = words[j],words[i]
        return " ".join(words)
    nc = []
    for clause in clauses:
        words = clause.split()
        if len(words) >= 4 and random.random() < 0.6:
            for _ in range(min(n, len(words)//2)):
                i,j = random.sample(range(len(words)),2)
                words[i],words[j] = words[j],words[i]
        nc.append(" ".join(words))
    return " ".join(nc)

def aug_conservative_deletion(text, p=0.07, seed=None):
    if seed is not None: random.seed(seed)
    words = text.split()
    if len(words) <= 6: return text
    nw = []
    for w in words:
        clean = w.lower().strip('.,!?;:\"\'()')
        if clean in STOPWORDS or len(clean)<=2: nw.append(w)
        elif random.random()>p: nw.append(w)
    return " ".join(nw) if len(nw)>=len(words)//2 else text

def aug_phrase_paraphrase(text, seed=None):
    if seed is not None: random.seed(seed)
    tl = text.lower()
    cands = [(p,r) for p,r in EVASION_PHRASES.items() if p in tl]
    if not cands: return aug_synonym_replace(text, n=2, seed=seed)
    chosen = random.sample(cands, min(random.randint(1,2), len(cands)))
    result = text
    for phrase,replacement in chosen:
        idx = result.lower().find(phrase)
        if idx >= 0:
            if idx==0 or result[idx-1] in '.!?':
                replacement=replacement[0].upper()+replacement[1:]
            result=result[:idx]+replacement+result[idx+len(phrase):]
    return result

def aug_combined_chain(text, seed=None):
    if seed is not None: random.seed(seed)
    return aug_structure_swap(aug_synonym_replace(text,n=2,seed=seed),n=1,seed=seed+1)

AUG_OPS = [aug_synonym_replace,aug_structure_swap,aug_conservative_deletion,
           aug_phrase_paraphrase,aug_combined_chain]

def augment_pe_samples(questions, answers, labels, target_label_id=1,
                       augment_factor=8, seed=42):
    aug_q,aug_a,aug_l = [],[],[]
    pe_idx = [i for i,l in enumerate(labels) if l==target_label_id]
    print(f"  PE samples:{len(pe_idx)}  factor:{augment_factor}x  -> +{len(pe_idx)*augment_factor} samples")
    for i,idx in enumerate(pe_idx):
        q,a,l = questions[idx],answers[idx],labels[idx]
        for k in range(augment_factor):
            op = AUG_OPS[k%len(AUG_OPS)]
            aug_q.append(q); aug_a.append(op(a,seed+i*500+k)); aug_l.append(l)
    return aug_q,aug_a,aug_l

print("Augmentation module ready.")

Augmentation module ready.


In [4]:
def map_evasion_label(raw):
    """
    Maps the actual CSV label strings to 3 coarse classes.

    Confirmed label values from the dataset:
      Non-Evasive      <- "Explicit", "Implicit"
                          (politician directly addresses the question)
      Partially Evasive <- "Partial/half-answer", "Clarification"
                          (partial or hedged answer given)
      Evasive          <- "Dodging", "General", "Deflection",
                          "Declining to answer", "Claims ignorance"
                          (response avoids/sidesteps the question)
    """
    raw = str(raw).strip()
    rl  = raw.lower()
    # Non-Evasive: direct replies
    if rl in ("explicit", "implicit"):
        return "Non-Evasive"
    # Partially Evasive: partial or clarifying responses
    if rl in ("partial/half-answer", "clarification") or "partial" in rl or "half" in rl:
        return "Partially Evasive"
    # Evasive: all avoidance strategies
    return "Evasive"

def load_evasion(filepath):
    print("Loading QA Evasion...")
    df = pd.read_csv(filepath)
    df["question"] = df["interview_question"].fillna("").str.strip()
    df["answer"]   = df["interview_answer"].fillna("").str.strip()

    # Print actual unique label values so you can verify mapping
    print(f"  Unique raw labels ({df['label'].nunique()}):")
    for v, cnt in df["label"].value_counts().items():
        mapped = map_evasion_label(v)
        print(f"    {str(v):<45} -> {mapped} (n={cnt})")

    df["coarse_label"] = df["label"].apply(map_evasion_label)
    df["label_id"]     = df["coarse_label"].map(EVASION_LABEL2ID).astype(int)
    df = df.dropna(subset=["question","answer","label_id"]).reset_index(drop=True)
    print(f"  Total:{len(df)}")
    for lbl in EVASION_LABELS:
        n = (df["coarse_label"]==lbl).sum()
        print(f"  {lbl:22s}: {n:4d} ({n/len(df)*100:.1f}%)")

    # Validate all 3 classes present (needed for stratified split + class weights)
    present = df["label_id"].unique()
    for lid, lbl in enumerate(EVASION_LABELS):
        if lid not in present:
            raise ValueError(
                f"Class '{lbl}' (id={lid}) has 0 samples after mapping.\n"
                f"Check the raw label values printed above and fix map_evasion_label().")

    idx=list(range(len(df))); lbl=df["label_id"].tolist()
    itr,itmp,_,ytmp = train_test_split(idx,lbl,test_size=0.20,random_state=SEED,stratify=lbl)
    iva,ite,_,_     = train_test_split(itmp,ytmp,test_size=0.50,random_state=SEED,stratify=ytmp)
    q,a,l = df["question"].tolist(),df["answer"].tolist(),df["label_id"].tolist()
    def ex(ii): return [q[i] for i in ii],[a[i] for i in ii],[l[i] for i in ii]
    tr,va,te = ex(itr),ex(iva),ex(ite)
    print(f"  Train:{len(itr)}  Val:{len(iva)}  Test:{len(ite)}")
    return tr,va,te

def load_stance_split(filepaths, split_name):
    dfs=[]
    for fp in filepaths:
        if os.path.exists(fp): dfs.append(pd.read_csv(fp))
    df=pd.concat(dfs,ignore_index=True)
    df=df[df["Stance"].isin(STANCE_LABEL2ID)].reset_index(drop=True)
    df["label_id"]=df["Stance"].map(STANCE_LABEL2ID).astype(int)
    df["target"]=df["Target"].fillna("").str.strip()
    df["tweet"]=df["Tweet"].fillna("").str.strip()
    df=df.dropna(subset=["target","tweet","label_id"]).reset_index(drop=True)
    return df["target"].tolist(),df["tweet"].tolist(),df["label_id"].tolist()

print("="*60)
(Qetr,Aetr,yetr),(Qeva,Aeva,yeva),(Qete,Aete,yete) = load_evasion(EVASION_CSV)

print()
print("Augmenting PE class in training set (8x)...")
aug_q,aug_a,aug_l = augment_pe_samples(Qetr,Aetr,yetr,target_label_id=1,augment_factor=8)
Qetr_aug=Qetr+aug_q; Aetr_aug=Aetr+aug_a; yetr_aug=yetr+aug_l
combined=list(zip(Qetr_aug,Aetr_aug,yetr_aug))
random.seed(SEED); random.shuffle(combined)
Qetr_aug,Aetr_aug,yetr_aug=[list(x) for x in zip(*combined)]
print(f"  Augmented train:{len(yetr_aug)}")
for lid,lbl in enumerate(EVASION_LABELS):
    n=sum(1 for y in yetr_aug if y==lid)
    print(f"  {lbl:22s}: {n:4d} ({n/len(yetr_aug)*100:.1f}%)")

print()
Qstr,Astr,ystr = load_stance_split(STANCE_TRAIN_CSVS,"train")
Qsva,Asva,ysva = load_stance_split(STANCE_VAL_CSVS,"val")
Qste,Aste,yste = load_stance_split(STANCE_TEST_CSVS,"test")
print(f"Stance -> train:{len(ystr)}  val:{len(ysva)}  test:{len(yste)}")
print("="*60)

ew = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1,2]),y=yetr_aug),dtype=torch.float).to(device)
sw = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1]),  y=ystr),dtype=torch.float).to(device)
print(f"Evasion weights: {[round(x,4) for x in ew.tolist()]}")
print(f"Stance  weights: {[round(x,4) for x in sw.tolist()]}")

Loading QA Evasion...
  Unique raw labels (9):
    Explicit                                      -> Non-Evasive (n=1052)
    Dodging                                       -> Evasive (n=706)
    Implicit                                      -> Non-Evasive (n=488)
    General                                       -> Evasive (n=386)
    Deflection                                    -> Evasive (n=381)
    Declining to answer                           -> Evasive (n=145)
    Claims ignorance                              -> Evasive (n=119)
    Clarification                                 -> Partially Evasive (n=92)
    Partial/half-answer                           -> Partially Evasive (n=79)
  Total:3448
  Non-Evasive           : 1540 (44.7%)
  Partially Evasive     :  171 (5.0%)
  Evasive               : 1737 (50.4%)
  Train:2758  Val:345  Test:345

Augmenting PE class in training set (8x)...
  PE samples:137  factor:8x  -> +1096 samples
  Augmented train:3854
  Non-Evasive           : 1232

In [5]:
tokenizer = DistilBertTokenizer.from_pretrained(BERT_MODEL)

def split_into_clauses(text):
    """
    Split answer into clauses. Safety guarantees:
      - Strip whitespace from every part
      - Filter empty strings AND parts with ≤5 chars (noise)
      - Always return at least 1 clause (fallback = full text, truncated)
      - Never return more than MAX_CLAUSES
    """
    text = text.strip() or "empty"
    parts = re.split(
        r'(?<=[.!?])\s+|(?:\s+(?:but|however|although|whereas|while|yet|'
        r'because|since|so|therefore|nevertheless|nonetheless)\s+)', text)
    # Strip each part and remove empties / too-short fragments
    parts = [p.strip() for p in parts if p.strip() and len(p.strip()) > 5]
    # Safety: always at least one clause
    if not parts:
        parts = [text[:CLAUSE_LEN * 2]]   # fallback: raw text, bounded
    # Hard cap at MAX_CLAUSES
    return parts[:MAX_CLAUSES]

class TaskDataset(Dataset):
    def __init__(self, ta, tb, labels, max_len, task_id):
        self.ta,self.tb,self.labels,self.max_len,self.task_id=ta,tb,labels,max_len,task_id
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc  = tokenizer(self.ta[idx],self.tb[idx],truncation=True,
                         padding="max_length",max_length=self.max_len,return_tensors="pt")
        # Change 4: encode question at full token level (not just for mean-pool)
        qenc = tokenizer(self.ta[idx],truncation=True,
                         padding="max_length",max_length=Q_MAX_LEN,return_tensors="pt")
        clauses = split_into_clauses(self.tb[idx]) if self.task_id==0 else [self.tb[idx]]
        ids_l,masks_l=[],[]
        for c in clauses[:MAX_CLAUSES]:
            ce=tokenizer(self.ta[idx],c,truncation=True,
                         padding="max_length",max_length=CLAUSE_LEN,return_tensors="pt")
            ids_l.append(ce["input_ids"].squeeze(0))
            masks_l.append(ce["attention_mask"].squeeze(0))
        n_real=len(ids_l)
        while len(ids_l)<MAX_CLAUSES:
            ids_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
            masks_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
        return {
            "input_ids":       enc["input_ids"].squeeze(0),
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "q_input_ids":     qenc["input_ids"].squeeze(0),
            "q_attention_mask":qenc["attention_mask"].squeeze(0),
            "clause_ids":      torch.stack(ids_l),
            "clause_masks":    torch.stack(masks_l),
            "n_clauses":       torch.tensor(n_real,dtype=torch.long),
            "labels":          torch.tensor(self.labels[idx],dtype=torch.long),
            "task_id":         torch.tensor(self.task_id,dtype=torch.long),
        }

def make_loader(ta,tb,y,tid,ml,shuffle):
    return DataLoader(TaskDataset(ta,tb,y,ml,tid),batch_size=BATCH_SIZE,
                      shuffle=shuffle,num_workers=2,pin_memory=True)

loaders = {
    "evasion_train": make_loader(Qetr_aug,Aetr_aug,yetr_aug,0,MAX_LEN,True),
    "evasion_val":   make_loader(Qeva,Aeva,yeva,0,MAX_LEN,False),
    "evasion_test":  make_loader(Qete,Aete,yete,0,MAX_LEN,False),
    "stance_train":  make_loader(Qstr,Astr,ystr,1,STANCE_MAX_LEN,True),
    "stance_val":    make_loader(Qsva,Asva,ysva,1,STANCE_MAX_LEN,False),
    "stance_test":   make_loader(Qste,Aste,yste,1,STANCE_MAX_LEN,False),
}
for k,v in loaders.items():
    print(f"  {k:<20} {len(v.dataset):>7,} samples  {len(v):>4} batches")
print("DataLoaders ready.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  evasion_train          3,854 samples   241 batches
  evasion_val              345 samples    22 batches
  evasion_test             345 samples    22 batches
  stance_train          17,224 samples  1077 batches
  stance_val             2,193 samples   138 batches
  stance_test            2,157 samples   135 batches
DataLoaders ready.


In [6]:
def mean_pooling(h, mask):
    m = mask.unsqueeze(-1).expand(h.size()).float()
    return torch.sum(h*m,1)/torch.clamp(m.sum(1),min=1e-9)

# ── Change 2: Focal Loss ──────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Numerically stable implementation:
      1. NO label smoothing — mixing smoothing with focal loss destabilises
         p_t computation (smoothed targets reduce p_t artificially, causing
         (1-p_t)^gamma to over-weight easy examples — the opposite of focal).
      2. p_t clamped min=1e-8 before log — prevents log(0) = -inf -> NaN.
      3. Works directly on hard (integer) targets via gather.
    """
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, logits, targets):
        # Standard softmax probabilities — no label smoothing
        p   = F.softmax(logits, dim=-1)                          # [B, K]
        # p_t: probability assigned to the TRUE class
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)       # [B]
        p_t = torch.clamp(p_t, min=1e-8)                        # prevent log(0)
        # Focal weight: down-weights easy (high p_t) examples
        focal_w = (1.0 - p_t) ** self.gamma                     # [B]
        # Cross-entropy for the true class only
        ce_loss = -torch.log(p_t)                                # [B]
        loss    = focal_w * ce_loss                              # [B]
        # Apply class weights as alpha (per-sample)
        if self.weight is not None:
            alpha = self.weight[targets]                         # [B]
            loss  = loss * alpha
        return loss.mean()

# ── Change 4: Token-Level Cross-Attention ────────────────────────────────
class TokenCrossAttention(nn.Module):
    """
    Cross-attention where question *tokens* (not mean-pool) attend over
    the full answer hidden states.

    Replaces the mean-pooled q_rep used in V3-C's gating, giving the
    model a richer, position-aware question signal.

    q_seq  : [B, Q, H]  — question token hidden states
    a_seq  : [B, T, H]  — full answer hidden states
    a_mask : [B, T]     — answer attention mask
    Returns: [B, H]     — question-attended answer summary
    """
    def __init__(self, H, n_heads=4, dropout=0.1):
        super().__init__()
        assert H % n_heads == 0
        self.n_heads  = n_heads
        self.d_head   = H // n_heads
        self.Wq = nn.Linear(H, H); self.Wk = nn.Linear(H, H); self.Wv = nn.Linear(H, H)
        self.out_proj = nn.Linear(H, H)
        self.norm     = nn.LayerNorm(H)
        self.drop     = nn.Dropout(dropout)
        # Save weights for Task 3 explanation
        self.last_attn_weights = None

    def forward(self, q_seq, a_seq, a_mask, q_mask=None):
        B, Q, H = q_seq.size()
        T       = a_seq.size(1)
        # Project
        Q_ = self.Wq(q_seq).view(B, Q, self.n_heads, self.d_head).transpose(1,2)  # [B,h,Q,d]
        K_ = self.Wk(a_seq).view(B, T, self.n_heads, self.d_head).transpose(1,2)  # [B,h,T,d]
        V_ = self.Wv(a_seq).view(B, T, self.n_heads, self.d_head).transpose(1,2)  # [B,h,T,d]
        scale  = self.d_head ** 0.5
        scores = torch.matmul(Q_, K_.transpose(-2,-1)) / scale                     # [B,h,Q,T]
        # Mask padding in answer
        if a_mask is not None:
            pad_mask = (a_mask == 0).unsqueeze(1).unsqueeze(2)                     # [B,1,1,T]
            scores   = scores.masked_fill(pad_mask, float("-inf"))
        attn = F.softmax(scores, dim=-1)                                            # [B,h,Q,T]
        attn = self.drop(attn)
        # Save averaged head weights for Task 3
        self.last_attn_weights = attn.mean(dim=1).detach()                         # [B,Q,T]
        ctx  = torch.matmul(attn, V_)                                              # [B,h,Q,d]
        ctx  = ctx.transpose(1,2).contiguous().view(B, Q, H)                      # [B,Q,H]
        ctx  = self.out_proj(ctx)
        # Mean-pool across question positions to get a single [B,H] vector
        if q_mask is not None:
            qm  = q_mask.unsqueeze(-1).float()
            out = (ctx * qm).sum(1) / qm.sum(1).clamp(min=1e-9)
        else:
            out = ctx.mean(1)
        return self.norm(out)

# ── Change 1: V3-C base (TokenLevelGating, no Graph, no ClausePool) ──────
class TokenLevelGating(nn.Module):
    """
    Token-level gating (from ablation V3-C).
    Each answer token is gated by the question representation,
    then mean-pooled. This is the strongest single component
    from ablation (E.F1 0.5055, PE 0.27).
    """
    def __init__(self, H, dr):
        super().__init__()
        self.Wa   = nn.Linear(H, H, bias=False)
        self.Wq   = nn.Linear(H, H, bias=True)
        self.drop = nn.Dropout(dr)

    def forward(self, h_seq, attention_mask, q_rep):
        # q_rep: [B, H] — can be mean-pooled OR token-cross-attn output
        gate    = torch.sigmoid((self.Wa(h_seq) + self.Wq(q_rep).unsqueeze(1)) / 0.7)
        h_gated = self.drop(gate * h_seq)
        m       = attention_mask.unsqueeze(-1).expand(h_gated.size()).float()
        return torch.sum(h_gated * m, 1) / torch.clamp(m.sum(1), min=1e-9)

# ── Change 7: Contrastive auxiliary loss ─────────────────────────────────
class ContrastiveLoss(nn.Module):
    """
    Supervised NT-Xent contrastive loss on clause representations.

    Within each batch, clause reps from examples with the same evasion
    label are pulled together; those from different labels are pushed apart.
    This encourages the clause encoder to learn label-discriminative
    representations, particularly helping the minority PE class.

    Only applied to evasion batches.
    tau: temperature (default 0.1 — standard for supervised contrastive).
    """
    def __init__(self, tau=0.1):
        super().__init__()
        self.tau = tau

    def forward(self, embeddings, labels):
        """
        embeddings : [B, H] — one rep per sample (we use gated output)
        labels     : [B]    — evasion class labels
        Returns scalar loss, or 0 if batch too small.

        NaN fix: F.normalize() produces NaN when norm==0 (zero-vector embeddings
        from all-padding batches). Use manual safe normalize with eps instead.
        """
        B = embeddings.size(0)
        if B < 2: return torch.tensor(0.0, device=embeddings.device)
        # Safe normalize: avoids NaN when embedding norm is 0
        norms = embeddings.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        z = embeddings / norms                                          # [B, H]
        sim = torch.matmul(z, z.T) / self.tau                          # [B, B]
        # Mask diagonal (self-similarity)
        mask_diag = torch.eye(B, dtype=torch.bool, device=z.device)
        sim = sim.masked_fill(mask_diag, float("-inf"))
        # Positive mask: same label, different index
        label_mat = labels.unsqueeze(0) == labels.unsqueeze(1)         # [B, B]
        pos_mask  = label_mat & ~mask_diag
        # If no positives exist (all unique labels in batch), return 0
        if pos_mask.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device)
        log_probs = F.log_softmax(sim, dim=-1)
        log_probs = torch.clamp(log_probs, min=-100)                   # NaN fix: guard log(0)
        loss = -(log_probs * pos_mask.float()).sum() / pos_mask.float().sum()
        # Final NaN guard: return 0 if loss is still NaN (degenerate batch)
        if torch.isnan(loss):
            return torch.tensor(0.0, device=embeddings.device)
        return loss

class ClassHead(nn.Module):
    """
    Classification head with optional length normalisation feature.
    When len_feat is provided (a [B,1] scalar = log(answer_token_count)),
    it is concatenated to the representation before the first linear,
    helping the model NOT conflate response length with evasion.
    (Fix for inference failure mode: long diplomatic answers → Evasive)
    """
    def __init__(self, H, n, dr, use_len_feat=False):
        super().__init__()
        self.use_len_feat = use_len_feat
        in_dim = H + 1 if use_len_feat else H
        self.net=nn.Sequential(nn.Dropout(dr),nn.Linear(in_dim,256),nn.ReLU(),
                               nn.Dropout(dr),nn.Linear(256,n))
    def forward(self, x, len_feat=None):
        if self.use_len_feat and len_feat is not None:
            x = torch.cat([x, len_feat], dim=-1)
        return self.net(x)

# ── Main model: V3-C + TokenCrossAttn ────────────────────────────────────
class MultiTaskV3C(nn.Module):
    """
    V3-C + All 7 changes:
      - Base: TokenLevelGating (V3-C ablation winner)
      - Change 4: TokenCrossAttention replaces mean-pooled question repr
      - Change 2: FocalLoss for evasion
      - Change 7: ContrastiveLoss on gated representations

    Forward:
      1. Encode full [Q;A] → h_seq
      2. Encode Q tokens only → q_seq  (Change 4 input)
      3. TokenCrossAttention(q_seq, h_seq) → q_rep_rich  (Change 4)
      4. TokenLevelGating(h_seq, q_rep_rich) → gated_pool
      5. ClassHead(gated_pool) → logits
      6. FocalLoss(logits, labels)  (Change 2)
      7. ContrastiveLoss(gated_pool, labels)  (Change 7, evasion only)
    """
    def __init__(self, ew, sw):
        super().__init__()
        self.encoder   = DistilBertModel.from_pretrained(BERT_MODEL)
        H = self.encoder.config.hidden_size

        # Change 4: token-level cross-attention for question
        self.token_cross_attn = TokenCrossAttention(H, n_heads=4, dropout=0.1)

        # Change 1: V3-C gating (one per task for independent calibration)
        self.evasion_gate = TokenLevelGating(H, DROPOUT)
        self.stance_gate  = TokenLevelGating(H, DROPOUT)

        self.evasion_head = ClassHead(H, len(EVASION_LABELS), DROPOUT, use_len_feat=True)
        self.stance_head  = ClassHead(H, len(STANCE_LABELS),  DROPOUT, use_len_feat=False)

        # Change 2: Focal loss for evasion; CE for stance
        self.evasion_loss_fn = FocalLoss(weight=ew, gamma=FOCAL_GAMMA)
        # NOTE: No label smoothing for evasion — incompatible with focal loss
        self.stance_loss_fn  = nn.CrossEntropyLoss(weight=sw,
                                                    label_smoothing=LABEL_SMOOTH)

        # Change 7: contrastive loss
        self.contrastive_loss_fn = ContrastiveLoss(tau=CONTRASTIVE_TAU)

    def freeze_inactive_head(self, task):
        off = self.stance_head  if task==0 else self.evasion_head
        on  = self.evasion_head if task==0 else self.stance_head
        for p in off.parameters(): p.requires_grad=False
        for m in [on, self.encoder, self.token_cross_attn,
                  self.evasion_gate, self.stance_gate]:
            for p in m.parameters(): p.requires_grad=True

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad=True

    def forward(self, input_ids, attention_mask, task_id, labels=None,
                q_input_ids=None, q_attention_mask=None,
                clause_ids=None, clause_masks=None, n_clauses=None):
        B    = input_ids.size(0)
        task = int(task_id[0].item())

        # 1. Encode full [Q;A] sequence
        h_out   = self.encoder(input_ids=input_ids,
                               attention_mask=attention_mask).last_hidden_state  # [B,T,H]

        # 2. Encode question tokens only (Change 4)
        q_out   = self.encoder(input_ids=q_input_ids,
                               attention_mask=q_attention_mask).last_hidden_state  # [B,Q,H]

        # 3. Token-level cross-attention: Q tokens attend over A tokens (Change 4)
        q_rep   = self.token_cross_attn(q_out, h_out, attention_mask, q_attention_mask)  # [B,H]

        # 4. Gate answer tokens by the rich question representation
        gate_fn = self.evasion_gate if task==0 else self.stance_gate
        gated   = gate_fn(h_out, attention_mask, q_rep)                                 # [B,H]

        # 5. Classification
        # Length feature: log(num_non_padding_tokens) — helps distinguish verbosity from evasion
        len_feat = torch.log(attention_mask.float().sum(dim=1, keepdim=True).clamp(min=1))  # [B,1]
        if task == 0:
            logits = self.evasion_head(gated, len_feat=len_feat)
        else:
            logits = self.stance_head(gated)

        loss = None
        if labels is not None:
            cls_loss  = (self.evasion_loss_fn if task==0 else self.stance_loss_fn)(logits, labels)
            # Change 7: add contrastive loss for evasion batches only
            if task == 0:
                contr_loss = self.contrastive_loss_fn(gated, labels)
                loss       = cls_loss + CONTRASTIVE_W * contr_loss
            else:
                loss = cls_loss

        return loss, logits

print("Model V3-C + TokenCrossAttn + FocalLoss + ContrastiveLoss defined.")
print("  Changes applied:")
print("  [1] Base: V3-C (TokenLevelGating only)")
print(f"  [2] FocalLoss (gamma={FOCAL_GAMMA}) for evasion — NaN fix: log_p clamped min=-100")
print("  [4] TokenCrossAttention replaces mean-pooled q_rep")
print(f"  [7] ContrastiveLoss on gated reps (evasion only, w={CONTRASTIVE_W})")
print("  [FIX] ClassHead uses log(answer_length) scalar feature for evasion head")

Model V3-C + TokenCrossAttn + FocalLoss + ContrastiveLoss defined.
  Changes applied:
  [1] Base: V3-C (TokenLevelGating only)
  [2] FocalLoss (gamma=3.0) for evasion — NaN fix: log_p clamped min=-100
  [4] TokenCrossAttention replaces mean-pooled q_rep
  [7] ContrastiveLoss on gated reps (evasion only, w=0.1)
  [FIX] ClassHead uses log(answer_length) scalar feature for evasion head


In [7]:
@torch.no_grad()
def evaluate(model, loader, label_names):
    model.eval(); model.unfreeze_all()
    all_preds,all_labels,total_loss=[],[],0.0
    for batch in loader:
        loss,logits=model(
            batch["input_ids"].to(device),batch["attention_mask"].to(device),
            batch["task_id"].to(device),  batch["labels"].to(device),
            batch["q_input_ids"].to(device),batch["q_attention_mask"].to(device),
            batch["clause_ids"].to(device),batch["clause_masks"].to(device),
            batch["n_clauses"].to(device))
        if not (torch.isnan(loss) or torch.isinf(loss)):
            total_loss+=loss.item()
        all_preds.extend(torch.argmax(logits,1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
    return {
        "loss":  total_loss/len(loader),
        "acc":   accuracy_score(all_labels,all_preds),
        "f1":    f1_score(all_labels,all_preds,average="macro",zero_division=0),
        "prec":  precision_score(all_labels,all_preds,average="macro",zero_division=0),
        "rec":   recall_score(all_labels,all_preds,average="macro",zero_division=0),
        "report":classification_report(all_labels,all_preds,target_names=label_names,zero_division=0),
        "cm":    pd.DataFrame(
                   confusion_matrix(all_labels,all_preds,labels=list(range(len(label_names)))),
                   index=[f"True:{l}" for l in label_names],
                   columns=[f"Pred:{l}" for l in label_names]),
        "preds":all_preds,"labels":all_labels,
    }

def print_eval(m, title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(f"  Loss:{m['loss']:.4f}  Acc:{m['acc']*100:.2f}%  MacroF1:{m['f1']:.4f}  "
          f"Prec:{m['prec']:.4f}  Rec:{m['rec']:.4f}")
    print("\n  Per-class Report:")
    for line in m["report"].strip().split("\n"): print(f"  {line}")
    print("\n  Confusion Matrix:")
    print(m["cm"].to_string()); print("="*60)

def get_pe_f1(report_str):
    for line in report_str.split("\n"):
        if "Partially Evasive" in line:
            try: return float(line.split()[3])
            except: return 0.0
    return 0.0

def train_and_save(model, save_path):
    best_avg_f1=0.0; patience_ctr=0

    shared  = list(model.encoder.parameters()) + list(model.token_cross_attn.parameters())
    ev_task = list(model.evasion_head.parameters()) + list(model.evasion_gate.parameters())
    st_task = list(model.stance_head.parameters())  + list(model.stance_gate.parameters())

    optimizer=AdamW([{"params":ev_task,"lr":LR_EVASION},
                     {"params":st_task,"lr":LR_STANCE},
                     {"params":shared, "lr":LR_EVASION}],weight_decay=0.01)

    # Change 5: longer schedule (EPOCHS=7)
    n_steps=len(loaders["stance_train"])*4*EPOCHS  # 3:1 evasion sampling: 4 batches per stance batch
    scheduler=get_cosine_schedule_with_warmup(optimizer,int(WARMUP_RATIO*n_steps),n_steps)

    # Change 5: freeze for FREEZE_EPOCHS=3
    for p in model.encoder.parameters(): p.requires_grad=False
    print(f"  Params:{sum(p.numel() for p in model.parameters()):,}  "
          f"Sampling:2:1  Scheduler:Cosine  Patience:{PATIENCE}")
    print(f"  Freeze:{FREEZE_EPOCHS} epochs  LR_e:{LR_EVASION}  LR_s:{LR_STANCE}")
    print(f"  Will save best checkpoint to: {save_path}")

    for epoch in range(1,EPOCHS+1):
        if epoch==FREEZE_EPOCHS+1:
            for p in model.encoder.parameters(): p.requires_grad=True
            print(f"  Epoch {epoch}: BERT UNFROZEN.")
        model.train()
        e_cycle=itertools.cycle(list(loaders["evasion_train"]))
        combined=[]
        for sb in loaders["stance_train"]:
            combined.append(next(e_cycle)); combined.append(next(e_cycle))
            combined.append(next(e_cycle)); combined.append(sb)  # FIX: 3:1 evasion sampling
        total=len(combined)
        run_e=run_s=ne=ns=0; t0=time.time()
        pbar=tqdm(combined,total=total,desc=f"Ep {epoch}/{EPOCHS}",leave=True)

        for step,batch in enumerate(pbar,1):
            task=int(batch["task_id"][0].item())
            model.freeze_inactive_head(task)
            loss,_=model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["task_id"].to(device),
                batch["labels"].to(device),
                batch["q_input_ids"].to(device),
                batch["q_attention_mask"].to(device),
                batch["clause_ids"].to(device),
                batch["clause_masks"].to(device),
                batch["n_clauses"].to(device))
            # NaN/inf guard: skip batch if loss is degenerate
            if torch.isnan(loss) or torch.isinf(loss):
                optimizer.zero_grad()
                continue
            (loss/GRAD_ACCUM).backward()
            if task==0: run_e+=loss.item(); ne+=1
            else:       run_s+=loss.item(); ns+=1
            if step%GRAD_ACCUM==0 or step==total:
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
            if step%100==0 or step==total:
                avg_e=run_e/ne if ne else 0; avg_s=run_s/ns if ns else 0
                pbar.set_description(f"Ep {epoch} | E:{avg_e:.3f} S:{avg_s:.3f}")
        pbar.close()
        elapsed=(time.time()-t0)/60
        ev=evaluate(model,loaders["evasion_val"],EVASION_LABELS)
        sv=evaluate(model,loaders["stance_val"], STANCE_LABELS)
        avg=(ev["f1"]+sv["f1"])/2; pe=get_pe_f1(ev["report"]); is_best=avg>best_avg_f1
        print(f"  Ep{epoch} ({elapsed:.1f}m) E:{run_e/ne:.4f} S:{run_s/ns:.4f} | "
              f"Val E:{ev['f1']:.4f} S:{sv['f1']:.4f} Avg:{avg:.4f} PE:{pe:.2f} "
              f"{'BEST' if is_best else f'p:{patience_ctr+1}/{PATIENCE}'}")
        print("  Val per-class evasion:")
        for line in ev["report"].strip().split("\n"):
            if any(lbl in line for lbl in EVASION_LABELS):
                print(f"    {line.strip()}")
        if is_best:
            best_avg_f1=avg; patience_ctr=0; torch.save(model.state_dict(),save_path)
        else:
            patience_ctr+=1
            if patience_ctr>=PATIENCE: print(f"  Early stopping."); break
        print()

    model.load_state_dict(torch.load(save_path,map_location=device)); model.unfreeze_all()
    et=evaluate(model,loaders["evasion_test"],EVASION_LABELS)
    st=evaluate(model,loaders["stance_test"], STANCE_LABELS)
    print(f"  TEST: E:{et['f1']:.4f} S:{st['f1']:.4f} "
          f"Avg:{(et['f1']+st['f1'])/2:.4f} PE:{get_pe_f1(et['report']):.2f} "
          f"Acc:{et['acc']*100:.2f}%")
    return et,st

print("Training function ready.")

Training function ready.


In [8]:
# ─────────────────────────────────────────────────────────────
# TRAIN — V3-C + All 7 changes
# ─────────────────────────────────────────────────────────────
print("\n"+"="*65)
print("  TRAINING: V3-C + TokenCrossAttn + FocalLoss + ContrastiveLoss")
print("="*65)

model = MultiTaskV3C(ew, sw).to(device)
e_test, s_test = train_and_save(model, FULL_MODEL_PATH)

print_eval(e_test, "EVASION TEST — V3-C Final")
print_eval(s_test, "STANCE  TEST — V3-C Final")

# Save predictions
pd.DataFrame({"question":Qete,"answer":Aete,
    "true":[EVASION_LABELS[l] for l in e_test["labels"]],
    "pred":[EVASION_LABELS[p] for p in e_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,"preds_v3c_final_evasion.csv"),index=False)
pd.DataFrame({"target":Qste,"tweet":Aste,
    "true":[STANCE_LABELS[l] for l in s_test["labels"]],
    "pred":[STANCE_LABELS[p] for p in s_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,"preds_v3c_final_stance.csv"),index=False)

print(f"\nModel saved: {FULL_MODEL_PATH}")
sz=os.path.getsize(FULL_MODEL_PATH)/(1024*1024)
print(f"  Size: {sz:.1f} MB")


  TRAINING: V3-C + TokenCrossAttn + FocalLoss + ContrastiveLoss


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  Ep5 (32.6m) E:0.2474 S:0.4680 | Val E:0.4833 S:0.7673 Avg:0.6253 PE:0.29 BEST
  Val per-class evasion:
    Non-Evasive       0.55      0.73      0.63       154
    Partially Evasive       0.29      0.29      0.29        17
    Evasive       0.63      0.45      0.53       174



Ep 6/7:   0%|          | 0/4308 [00:00<?, ?it/s]

  Ep6 (32.6m) E:0.2423 S:0.4303 | Val E:0.4813 S:0.7606 Avg:0.6209 PE:0.29 p:1/3
  Val per-class evasion:
    Non-Evasive       0.54      0.73      0.62       154
    Partially Evasive       0.31      0.29      0.30        17
    Evasive       0.63      0.44      0.52       174



Ep 7/7:   0%|          | 0/4308 [00:00<?, ?it/s]

  Ep7 (32.5m) E:0.2407 S:0.4015 | Val E:0.4941 S:0.7691 Avg:0.6316 PE:0.29 BEST
  Val per-class evasion:
    Non-Evasive       0.58      0.67      0.62       154
    Partially Evasive       0.29      0.29      0.29        17
    Evasive       0.62      0.53      0.57       174

  TEST: E:0.5572 S:0.7688 Avg:0.6630 PE:0.41 Acc:60.29%

  EVASION TEST — V3-C Final
  Loss:0.6691  Acc:60.29%  MacroF1:0.5572  Prec:0.5737  Rec:0.5479

  Per-class Report:
  precision    recall  f1-score   support
  
        Non-Evasive       0.57      0.67      0.62       154
  Partially Evasive       0.50      0.41      0.45        17
            Evasive       0.65      0.56      0.60       174
  
           accuracy                           0.60       345
          macro avg       0.57      0.55      0.56       345
       weighted avg       0.61      0.60      0.60       345

  Confusion Matrix:
                        Pred:Non-Evasive  Pred:Partially Evasive  Pred:Evasive
True:Non-Evasive                  

In [9]:
# ─────────────────────────────────────────────────────────────
# TASK 3 — EXPLANATION SPAN EXTRACTION (Changes 3 + 4)
#
# Change 3: Two fixes over the plain attention baseline:
#   (a) Mismatch signal: highlight clauses where the answer
#       token distribution diverges most from the question tokens.
#       A high mismatch score = the answer "went off-topic" here.
#   (b) Class-conditional logic:
#       - Non-Evasive  : show highest-relevance clause (what IS addressed)
#       - Partially Evasive: show BOTH the most relevant AND the most
#         mismatched clause (partial answer + where it drifts)
#       - Evasive      : show the highest-mismatch clause (where evasion occurs)
#
# Change 4 benefit: token-level cross-attention weights [B, Q_len, T_len]
#   now give finer alignment signal per query token, not just per clause.
# ─────────────────────────────────────────────────────────────

def compute_mismatch_signal(token_attn_weights, q_seq_len, a_seq_len):
    """
    Compute per-answer-token mismatch from token cross-attention.
    token_attn_weights: [Q_len, T_len] — averaged across heads
    Returns: [T_len] mismatch scores (higher = answer token less attended by Q)
    """
    # Average attention each answer token received from question tokens
    # Shape: [T_len] — how much each answer position was attended to
    attn_from_q = token_attn_weights.mean(0)  # [T_len]
    # Mismatch = low attention from Q = the answer diverged from question context
    mismatch = 1.0 - attn_from_q
    return mismatch

@torch.no_grad()
def extract_explanation(model, question, answer):
    """
    Run a single Q+A pair and return:
    - evasion prediction + confidence
    - stance prediction + confidence
    - clause-level explanation (class-conditional, Change 3)
    - mismatch signal (Change 3a)
    """
    model.eval()
    enc  = tokenizer(question,answer,truncation=True,padding="max_length",
                     max_length=MAX_LEN,return_tensors="pt").to(device)
    qenc = tokenizer(question,truncation=True,padding="max_length",
                     max_length=Q_MAX_LEN,return_tensors="pt").to(device)
    # Clause tensors (needed for interface compatibility)
    clauses = split_into_clauses(answer)
    ids_l,masks_l=[],[]
    for c in clauses[:MAX_CLAUSES]:
        ce=tokenizer(question,c,truncation=True,padding="max_length",
                     max_length=CLAUSE_LEN,return_tensors="pt").to(device)
        ids_l.append(ce["input_ids"]); masks_l.append(ce["attention_mask"])
    n_real=len(ids_l)
    while len(ids_l)<MAX_CLAUSES:
        ids_l.append(torch.zeros(1,CLAUSE_LEN,dtype=torch.long,device=device))
        masks_l.append(torch.zeros(1,CLAUSE_LEN,dtype=torch.long,device=device))
    clause_ids   = torch.cat(ids_l).unsqueeze(0)
    clause_masks = torch.cat(masks_l).unsqueeze(0)
    n_clauses    = torch.tensor([n_real],dtype=torch.long,device=device)

    # Evasion forward
    _,e_logits=model(enc["input_ids"],enc["attention_mask"],
                     torch.tensor([0],device=device),
                     q_input_ids=qenc["input_ids"],
                     q_attention_mask=qenc["attention_mask"],
                     clause_ids=clause_ids,clause_masks=clause_masks,
                     n_clauses=n_clauses)
    evasion_pred = EVASION_LABELS[torch.argmax(e_logits,1).item()]
    evasion_conf = float(torch.softmax(e_logits,1).max())

    # Retrieve token-level cross-attention weights (Change 4)
    token_attn = model.token_cross_attn.last_attn_weights[0].cpu().numpy()  # [Q, T]

    # Stance forward
    enc_s  = tokenizer(question,answer,truncation=True,padding="max_length",
                       max_length=STANCE_MAX_LEN,return_tensors="pt").to(device)
    qenc_s = tokenizer(question,truncation=True,padding="max_length",
                       max_length=Q_MAX_LEN,return_tensors="pt").to(device)
    _,s_logits=model(enc_s["input_ids"],enc_s["attention_mask"],
                     torch.tensor([1],device=device),
                     q_input_ids=qenc_s["input_ids"],
                     q_attention_mask=qenc_s["attention_mask"],
                     clause_ids=clause_ids,clause_masks=clause_masks,
                     n_clauses=n_clauses)
    stance_pred = STANCE_LABELS[torch.argmax(s_logits,1).item()]
    stance_conf = float(torch.softmax(s_logits,1).max())

    # ── Change 3a: Mismatch signal per clause ──────────────────────────
    mismatch_scores = []
    relevance_scores = []
    for k, clause in enumerate(clauses):
        # Re-encode this clause with question
        ce = tokenizer(question, clause, truncation=True, padding="max_length",
                       max_length=CLAUSE_LEN, return_tensors="pt").to(device)
        q_out = model.encoder(input_ids=qenc["input_ids"],
                              attention_mask=qenc["attention_mask"]).last_hidden_state
        a_out = model.encoder(input_ids=ce["input_ids"],
                              attention_mask=ce["attention_mask"]).last_hidden_state
        # Clause-level relevance: mean of how much Q attends to clause tokens
        # Use stored token_attn (global) or per-clause Q-A similarity
        q_mean = q_out[0, :qenc["attention_mask"].sum().item()].mean(0)  # [H]
        a_mean = a_out[0, :ce["attention_mask"].sum().item()].mean(0)    # [H]
        relevance = float(F.cosine_similarity(q_mean.unsqueeze(0), a_mean.unsqueeze(0)))
        # Mismatch = 1 - relevance (how much this clause drifts from Q)
        mismatch  = 1.0 - max(0.0, relevance)
        relevance_scores.append(relevance)
        mismatch_scores.append(mismatch)

    # Normalise
    rel_arr = np.array(relevance_scores, dtype=float)
    mis_arr = np.array(mismatch_scores,  dtype=float)
    rel_arr = rel_arr / (rel_arr.sum() + 1e-9)
    mis_arr = mis_arr / (mis_arr.sum() + 1e-9)

    clause_data = list(zip(clauses, rel_arr.tolist(), mis_arr.tolist()))

    # ── Change 3b: Class-conditional top-clause selection ──────────────
    if evasion_pred == "Non-Evasive":
        # Show highest-relevance clause (what IS directly addressed)
        top_idx    = int(np.argmax(rel_arr))
        top_clause = clauses[top_idx]
        expl_note  = f"Most relevant clause (high Q-A alignment):"
    elif evasion_pred == "Partially Evasive":
        # Show BOTH: most relevant (the partial answer part) AND most mismatched
        rel_idx    = int(np.argmax(rel_arr))
        mis_idx    = int(np.argmax(mis_arr))
        top_clause = clauses[rel_idx]
        drift_clause = clauses[mis_idx] if mis_idx != rel_idx else "(same clause)"
        expl_note  = f"Partial answer clause | Drift clause: {drift_clause[:100]}"
    else:  # Evasive
        # Show highest-mismatch clause (where the evasion occurs)
        top_idx    = int(np.argmax(mis_arr))
        top_clause = clauses[top_idx]
        expl_note  = f"Most evasive clause (high Q-A mismatch):"

    return {
        "evasion_pred":    evasion_pred,
        "evasion_conf":    evasion_conf,
        "stance_pred":     stance_pred,
        "stance_conf":     stance_conf,
        "clause_data":     clause_data,      # [(clause, relevance, mismatch), ...]
        "top_clause":      top_clause,
        "expl_note":       expl_note,
        "token_attn":      token_attn,        # [Q_len, T_len] for fine-grained viz
    }

def visualize_explanation(question, answer, result):
    """HTML heatmap — dual-signal: green=relevant, red=mismatched."""
    evasion_color = {"Non-Evasive":"#4CAF50","Partially Evasive":"#FF9800","Evasive":"#F44336"}
    stance_color  = {"FAVOR":"#2196F3","AGAINST":"#9C27B0"}
    ec = evasion_color.get(result["evasion_pred"],"#999")
    sc = stance_color.get(result["stance_pred"],"#999")
    html = (
        "<div style='font-family:sans-serif;max-width:820px;padding:16px;"
        "border:1px solid #ddd;border-radius:8px;'>"
        f"<p style='font-size:13px;color:#666;margin:0 0 8px'><b>Question:</b> {question[:200]}</p>"
        "<div style='display:flex;gap:12px;margin-bottom:12px;'>"
        f"<span style='background:{ec};color:white;padding:4px 10px;"
        f"border-radius:4px;font-size:13px;font-weight:500;'>"
        f"{result['evasion_pred']} ({result['evasion_conf']*100:.0f}%)</span>"
        f"<span style='background:{sc};color:white;padding:4px 10px;"
        f"border-radius:4px;font-size:13px;font-weight:500;'>"
        f"{result['stance_pred']} ({result['stance_conf']*100:.0f}%)</span>"
        "</div>"
        "<p style='font-size:12px;color:#888;margin:0 0 8px'>"
        "Green tint = high Q-A relevance &nbsp;|&nbsp; Red tint = high mismatch (evasion signal)</p>"
        "<div style='line-height:2.0;font-size:14px;'>"
    )
    for clause, rel, mis in result["clause_data"]:
        green_a = min(0.75, rel * 3.0)
        red_a   = min(0.75, mis * 3.0)
        if result["evasion_pred"] == "Non-Evasive":
            bg = f"rgba(76,175,80,{green_a:.2f})"
        elif result["evasion_pred"] == "Evasive":
            bg = f"rgba(244,67,54,{red_a:.2f})"
        else:  # Partially Evasive — blend
            bg = f"rgba({int(76+168*mis)},{int(175-143*mis)},{int(80-46*mis)},{max(green_a,red_a):.2f})"
        html += f"<span style='background:{bg};padding:2px 5px;border-radius:3px;margin:2px;display:inline;'>{clause}</span> "
    html += "</div>"
    html += f"<p style='font-size:12px;color:#555;margin:8px 0 0'><b>{result['expl_note']}</b></p>"
    html += f"<p style='font-size:12px;color:#777;margin:4px 0 0'><i>{result['top_clause'][:160]}</i></p>"
    html += "</div>"
    display(HTML(html))

print("Task 3 — Explanation extraction ready (Changes 3 + 4 + LenFeat).")
print("  Mismatch signal: cosine divergence per clause (Q-A mean cosine similarity)")
print("  Class-conditional: Non-Evasive=relevance, PE=both, Evasive=mismatch")
print("  NOTE: Task 3 explanation is post-hoc heuristic (cosine sim), not a trained head.")
print("  The evasion head now uses log(answer_length) to reduce length-evasion conflation.")

Task 3 — Explanation extraction ready (Changes 3 + 4 + LenFeat).
  Mismatch signal: cosine divergence per clause (Q-A mean cosine similarity)
  Class-conditional: Non-Evasive=relevance, PE=both, Evasive=mismatch
  NOTE: Task 3 explanation is post-hoc heuristic (cosine sim), not a trained head.
  The evasion head now uses log(answer_length) to reduce length-evasion conflation.


In [10]:
# ─────────────────────────────────────────────────────────────
# RUN TASK 3 — Qualitative case studies
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  TASK 3 — EXPLANATION SPAN EXTRACTION")
print("="*60)

for target_label in ["Non-Evasive","Partially Evasive","Evasive"]:
    class_id = EVASION_LABEL2ID[target_label]
    correct_idx = [i for i,(t,p) in enumerate(zip(e_test["labels"],e_test["preds"]))
                   if t==class_id and p==class_id]
    wrong_idx   = [i for i,(t,p) in enumerate(zip(e_test["labels"],e_test["preds"]))
                   if t==class_id and p!=class_id]
    print(f"\n{'─'*50}")
    print(f"  Class: {target_label}  (correct:{len(correct_idx)}  wrong:{len(wrong_idx)})")
    print(f"{'─'*50}")
    for k,idx in enumerate(correct_idx[:2]):
        q=Qete[idx]; a=Aete[idx]
        result=extract_explanation(model,q,a)
        print(f"\nExample {k+1} (CORRECT):")
        visualize_explanation(q,a,result)
    if wrong_idx:
        idx=wrong_idx[0]; q=Qete[idx]; a=Aete[idx]
        result=extract_explanation(model,q,a)
        true_lbl=EVASION_LABELS[e_test["labels"][idx]]
        print(f"\nExample (WRONG — true: {true_lbl}):")
        visualize_explanation(q,a,result)

# Save all explanations
print("\nGenerating explanations for all test samples...")
rows=[]
for i,(q,a) in enumerate(zip(Qete,Aete)):
    try:
        res=extract_explanation(model,q,a)
        rows.append({
            "question":       q,
            "answer":         a,
            "true_evasion":   EVASION_LABELS[e_test["labels"][i]],
            "pred_evasion":   res["evasion_pred"],
            "evasion_conf":   round(res["evasion_conf"],4),
            "pred_stance":    res["stance_pred"],
            "stance_conf":    round(res["stance_conf"],4),
            "top_clause":     res["top_clause"],
            "expl_note":      res["expl_note"],
            "clause_details": str([(c[:80],round(r,4),round(m,4))
                                   for c,r,m in res["clause_data"]]),
        })
    except Exception as e:
        pass
df_expl=pd.DataFrame(rows)
ep=os.path.join(OUTPUT_DIR,"task3_explanations_v3c_final.csv")
df_expl.to_csv(ep,index=False)
print(f"  Saved {len(df_expl)} records to {ep}")


  TASK 3 — EXPLANATION SPAN EXTRACTION

──────────────────────────────────────────────────
  Class: Non-Evasive  (correct:103  wrong:51)
──────────────────────────────────────────────────

Example 1 (CORRECT):



Example 2 (CORRECT):



Example (WRONG — true: Non-Evasive):



──────────────────────────────────────────────────
  Class: Partially Evasive  (correct:7  wrong:10)
──────────────────────────────────────────────────

Example 1 (CORRECT):



Example 2 (CORRECT):



Example (WRONG — true: Partially Evasive):



──────────────────────────────────────────────────
  Class: Evasive  (correct:98  wrong:76)
──────────────────────────────────────────────────

Example 1 (CORRECT):



Example 2 (CORRECT):



Example (WRONG — true: Evasive):



Generating explanations for all test samples...
  Saved 345 records to /kaggle/working/task3_explanations_v3c_final.csv


In [11]:
# ─────────────────────────────────────────────────────────────
# FINAL RESULTS TABLE
# NOTE on label mapping: "Clarification" was added to Partially Evasive in this run,
# bumping PE from 79→171 examples. Mid-report Table 1 showed PE=79 (2.3%).
# This is a defensible choice (clarifications are partial answers) but reviewers
# should be aware the PE support doubled. To reproduce mid-report numbers, revert
# map_evasion_label() to exclude "Clarification" from the PE bucket.
# ─────────────────────────────────────────────────────────────
print("\n"+"="*78)
print("  FINAL RESULTS — ALL SUBMISSIONS + THIS RUN")
print("="*78)

results = {
    "Mid V1 (baseline)":           {"e":0.4592,"s":0.7651,"avg":0.6122,"pe":0.14},
    "Mid V2 (gating)":             {"e":0.4976,"s":0.7633,"avg":0.6304,"pe":0.27},
    "Mid V3-Full (no aug)":        {"e":0.4692,"s":0.7592,"avg":0.6142,"pe":0.13},
    "Ablation V3-C (gating only)": {"e":0.5055,"s":0.7669,"avg":0.6362,"pe":0.27},
    "Ablation V3-A (no graph)":    {"e":0.4462,"s":0.7694,"avg":0.6078,"pe":0.13},
    "Ablation V3-D (no gating)":   {"e":0.4495,"s":0.7524,"avg":0.6010,"pe":0.10},
    "FINAL V3-C + All Changes":    {"e":e_test["f1"],"s":s_test["f1"],
                                    "avg":(e_test["f1"]+s_test["f1"])/2,
                                    "pe":get_pe_f1(e_test["report"])},
}

best_avg=max(v["avg"] for v in results.values())
print(f"  {'Variant':<44} {'E.F1':>7} {'S.F1':>7} {'Avg':>7} {'PE':>6}")
print(f"  {'─'*72}")
for name,r in results.items():
    mark=" <- BEST" if abs(r["avg"]-best_avg)<1e-4 else ""
    print(f"  {name:<44} {r['e']:>7.4f} {r['s']:>7.4f} {r['avg']:>7.4f} {r['pe']:>6.2f}{mark}")
print(f"  {'='*72}")

pd.DataFrame([{"variant":k,"e_f1":v["e"],"s_f1":v["s"],"avg_f1":v["avg"],"pe_f1":v["pe"]}
              for k,v in results.items()
]).to_csv(os.path.join(OUTPUT_DIR,"final_results_all.csv"),index=False)

print(f"\nFiles saved in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    sz=os.path.getsize(os.path.join(OUTPUT_DIR,f))
    unit="MB" if sz>100000 else "KB"; val=sz/(1024*1024) if sz>100000 else sz/1024
    print(f"  {f:<50} {val:.1f} {unit}")


  FINAL RESULTS — ALL SUBMISSIONS + THIS RUN
  Variant                                         E.F1    S.F1     Avg     PE
  ────────────────────────────────────────────────────────────────────────
  Mid V1 (baseline)                             0.4592  0.7651  0.6122   0.14
  Mid V2 (gating)                               0.4976  0.7633  0.6304   0.27
  Mid V3-Full (no aug)                          0.4692  0.7592  0.6142   0.13
  Ablation V3-C (gating only)                   0.5055  0.7669  0.6362   0.27
  Ablation V3-A (no graph)                      0.4462  0.7694  0.6078   0.13
  Ablation V3-D (no gating)                     0.4495  0.7524  0.6010   0.10
  FINAL V3-C + All Changes                      0.5572  0.7688  0.6630   0.41 <- BEST

Files saved in /kaggle/working:
  __notebook__.ipynb                                 0.3 MB
  best_model_v3c_final.pt                            272.7 MB
  final_results_all.csv                              0.4 KB
  preds_v3c_final_evasion.csv   

In [12]:
# ═══════════════════════════════════════════════════════════
# INFERENCE — Load saved model and predict on new examples
# ═══════════════════════════════════════════════════════════

def load_model_for_inference(model_path, device):
    dummy_ew = torch.ones(3).to(device)
    dummy_sw = torch.ones(2).to(device)
    m = MultiTaskV3C(dummy_ew, dummy_sw).to(device)
    m.load_state_dict(torch.load(model_path, map_location=device))
    m.eval()
    print(f"Model loaded from {model_path}")
    print(f"  Params: {sum(p.numel() for p in m.parameters()):,}")
    return m

def predict(model, question, answer):
    return extract_explanation(model, question, answer)

def predict_batch(model, questions, answers):
    results=[]
    for q,a in tqdm(zip(questions,answers),total=len(questions),desc="Predicting"):
        try: results.append(predict(model,q,a))
        except Exception as e: results.append({"evasion_pred":"Error","error":str(e)})
    return results

print("Loading saved model for inference demo...")
inference_model = load_model_for_inference(FULL_MODEL_PATH, device)

demo_questions=Qeva[:5]; demo_answers=Aeva[:5]
demo_true=[EVASION_LABELS[y] for y in yeva[:5]]
print("\nRunning inference on 5 validation examples:")
print("─"*60)
for i,(q,a,true_lbl) in enumerate(zip(demo_questions,demo_answers,demo_true)):
    result=predict(inference_model,q,a)
    correct="✓" if result["evasion_pred"]==true_lbl else "✗"
    print(f"\nExample {i+1} {correct}")
    print(f"  Q: {q[:100]}...")
    print(f"  True:  {true_lbl}")
    print(f"  Pred:  {result['evasion_pred']} ({result['evasion_conf']*100:.0f}%) | "
          f"Stance: {result['stance_pred']} ({result['stance_conf']*100:.0f}%)")
    print(f"  {result['expl_note']}")
    print(f"  => {result['top_clause'][:120]}")

print("\n" + "─"*60)
print("Inference complete.")

Loading saved model for inference demo...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded from /kaggle/working/best_model_v3c_final.pt
  Params: 71,482,885

Running inference on 5 validation examples:
────────────────────────────────────────────────────────────

Example 1 ✓
  Q: Q. Thank you, Mr. President. Today's fix that you just announced leaves it up to State insurance com...
  True:  Evasive
  Pred:  Evasive (76%) | Stance: AGAINST (76%)
  Most evasive clause (high Q-A mismatch):
  => All right.

Example 2 ✓
  Q: Q. Thank you, Mr. President. We're coming up on the 1-year anniversary of the killing of bin Laden. ...
  True:  Non-Evasive
  Pred:  Non-Evasive (66%) | Stance: FAVOR (76%)
  Most relevant clause (high Q-A alignment):
  => First of all, Christi, I hardly think that you've seen any excessive celebration taking place here.

Example 3 ✓
  Q: Q. It's in the meantime, sir, that I'm curious about. As you just said, raising the debt ceiling is ...
  True:  Evasive
  Pred:  Evasive (54%) | Stance: FAVOR (55%)
  Most evasive clause (high Q-A mismatch):
 